<a href="https://colab.research.google.com/github/Stronghold1388/Agent_experiments/blob/main/Lang_practice(module_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-openai huggingface_hub
#langchain|core framework
#langchain-openai|integration layer that lets langchain talk to openai (only needed for openai for other we dont need)
#hugginface|hosting the model


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3


In [2]:
from google.colab import userdata
#langchain provides wrapper to use its ecosystem
from huggingface_hub.inference._providers import openai
from langchain_openai.chat_models import ChatOpenAI
from huggingface_hub import InferenceClient
import os

# initializing Hugging face client
client= InferenceClient(model="openai/gpt-oss-120b")
#wrap it in Lancchain's chatopenai interface
model=ChatOpenAI(
    openai_api_key=userdata.get('HF_TOKEN'),
    model_name="openai/gpt-oss-120b",
    base_url="https://router.huggingface.co/v1"
)

In [3]:
response=model.invoke("what is the capitak of moon?")
print(response.content)#Direct Model Call| input are raw string|output is simpler| no conversation history, thus do not need Indexing
# in this case interpretation can also differ

The Moon isn’t a country, so it doesn’t have an official “capital” in the way nations on Earth do.  

**Why there’s no capital**

| Reason | Explanation |
|--------|-------------|
| **No sovereign government** | The Moon is considered a **global commons** under the 1967 Outer Space Treaty, which says no nation can claim ownership of celestial bodies. |
| **No permanent civilian population** | All human activity on the Moon so far has been short‑term missions (Apollo, robotic landers, etc.). There’s no permanent city or administrative hub. |
| **International jurisdiction** | Space activities are overseen by international agreements (Outer Space Treaty, Moon Agreement) rather than a single nation’s laws. |

---

### What *is* being planned for the Moon?

While there’s no capital today, several space agencies and private companies have announced plans for long‑term lunar outposts that could become de‑facto “centers” of activity:

| Project | Operator | Planned Site | Status (mid‑2026) |


In [ ]:
from pprint import pprint
pprint(response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-096b963c-9b8d-478a-801d-83bf30c88dbb',
 'logprobs': None,
 'model_name': 'gpt-oss-120b',
 'model_provider': 'openai',
 'system_fingerprint': 'fp_46358675aa6b1ba4d98d',
 'token_usage': {'completion_tokens': 649,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': None,
                                               'reasoning_tokens': 136,
                                               'rejected_prediction_tokens': 0},
                 'prompt_tokens': 75,
                 'prompt_tokens_details': {'audio_tokens': None,
                                           'cached_tokens': 0},
                 'total_tokens': 724}}


Multi conversation agent

In [8]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
agent= create_agent(model=model)
#HumanMessage=Its langchain's agent abstraction| it internally manages converstion state,indexing, response formatting.
#agent is designed to simulate a chatloop
question=HumanMessage(content="What is the capital of the moon?")
result=agent.invoke(
  {"messages":[question]})

print(result['messages'][1].content)# indexing is [1]as it shows user output rather that user input which is in [0]| in case of multiple back-forth indexing can require 3,4... so on.


The Moon doesn’t have an official capital—​it’s a natural satellite, not a nation.  
There’s no government, no cities, and thus no capital city in the real world.

(If you’re thinking of fiction, different stories have imagined “moon capitals” of their own—​for example, the 1995 TV series *Space: Above and Beyond* featured “Lunar Base” as a hub, and some science‑fiction novels talk about “Luna City” or “Tranquility City.” But in reality, the Moon is simply a rocky world orbiting Earth, with no capital.)


In [10]:
from pprint import pprint
pprint(result)

{'messages': [HumanMessage(content='What is the capital of the moon?', additional_kwargs={}, response_metadata={}, id='3f173f03-8e4e-4840-be0b-18421774334a'),
              AIMessage(content='The Moon doesn’t have an official capital—\u200bit’s a natural satellite, not a nation.  \nThere’s no government, no cities, and thus no capital city in the real world.\n\n(If you’re thinking of fiction, different stories have imagined “moon capitals” of their own—\u200bfor example, the 1995 TV series *Space: Above and Beyond* featured “Lunar Base” as a hub, and some science‑fiction novels talk about “Luna City” or “Tranquility City.” But in reality, the Moon is simply a rocky world orbiting Earth, with no capital.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 210, 'prompt_tokens': 75, 'total_tokens': 285, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 75, 'rejected_prediction_tokens': 0}, '

In [13]:
from langchain.messages import AIMessage, HumanMessage

result=agent.invoke(  #invoke provide entire answer| while .stream() provide chunks of answer
    {'messages':[
        HumanMessage(content='what is the capital of Moon?'),
        AIMessage(content='The capital of the Moon is Luna city.'),
        HumanMessage(content='Interseting, tell me more about Luna')
    ]}
)
pprint(result)

{'messages': [HumanMessage(content='what is the capital of Moon?', additional_kwargs={}, response_metadata={}, id='9b702238-2a4b-4ffb-a549-5ed236f3fe8b'),
              AIMessage(content='The capital of the Moon is Luna city.', additional_kwargs={}, response_metadata={}, id='6fbe031e-bbfa-4457-b314-bc02b550ca41', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interseting, tell me more about Luna', additional_kwargs={}, response_metadata={}, id='7f9d5ea4-2396-44df-8e3a-41a708b479e3'),
              AIMessage(content='There isn’t an official “capital” of the Moon—Earth‑orbiting bodies don’t have governments in the way nations do. \u202fBut the idea of a lunar city that could serve as a hub for science, industry, tourism, and even governance has been a favorite of scientists, engineers, and sci‑fi writers for decades. \u202fBelow is an overview of the most talked‑about projects, concepts, and the practical challenges that would shape any future “Luna City.”  \n